# Gold Product Dimension

## Objective
Build a business-ready product dimension from the Silver Products dataset.

## Responsibilities
- Load accepted Silver product data.
- Select descriptive and analytical product attributes.
- Add Gold processing metadata.
- Validate schema, row count, and key integrity.
- Write the Gold dimension to Parquet and verify the saved output.

**Grain:** One row per product.
**Primary key:** `product_id`.

### Configuration / Paths

In [1]:
from google.colab import drive
drive.mount("/content/drive")

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"

SILVER_PRODUCT_PATH = f"{FILE_PATH}/silver/products"
GOLD_PRODUCT_PATH = f"{FILE_PATH}/gold/dim_product"

EXPECTED_PRODUCT_COUNT = 996

Mounted at /content/drive


### Spark Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("gold-dim-product")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


### Load Silver Products

In [3]:
df_silver_product = spark.read.parquet(
    SILVER_PRODUCT_PATH
)

silver_product_count = df_silver_product.count()

print("Silver product count:", silver_product_count)

Silver product count: 996


### Inspect Source Data

In [4]:
df_silver_product.printSchema()

df_silver_product.show(10, truncate=False)

print("Available columns:")
print(df_silver_product.columns)

# Inspect the product status and category values before modeling.
df_silver_product.groupBy("is_active").count().show()
df_silver_product.groupBy("category").count().show()
df_silver_product.groupBy("subcategory").count().show()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- cost_price: decimal(10,2) (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- _silver_processed_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+----------+---------------------+------------+------+--------------+-----------+---------+----------+----------+---------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+
|product_id|product_name         |created_date|brand |category      |subcategory|is_active|cost_price|unit_price|_silver_processed_timestamp|_source_file                                                           

### Select Gold Product Attributes

In [5]:
GOLD_PRODUCT_COLUMNS = [
    "product_id",
    "product_name",
    "brand",
    "category",
    "subcategory",
    "created_date",
    "is_active",
    "cost_price",
    "unit_price",
]

df_gold_product = df_silver_product.select(
    *GOLD_PRODUCT_COLUMNS
)

df_gold_product.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- is_active: string (nullable = true)
 |-- cost_price: decimal(10,2) (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)



### Add Gold Audit Metadata

In [6]:
df_gold_product = df_gold_product.withColumn(
    "_gold_processed_timestamp",
    F.current_timestamp()
)

df_gold_product.show(10, truncate=False)

+----------+---------------------+------+--------------+-----------+------------+---------+----------+----------+--------------------------+
|product_id|product_name         |brand |category      |subcategory|created_date|is_active|cost_price|unit_price|_gold_processed_timestamp |
+----------+---------------------+------+--------------+-----------+------------+---------+----------+----------+--------------------------+
|P00001    |Orion Wellness 0001  |Nova  |Beauty        |Wellness   |2025-05-01  |true     |14.01     |22.83     |2026-10-29 12:25:12.225509|
|P00002    |Atlas Laptops 0002   |Orion |Electronics   |Laptops    |2024-09-06  |true     |140.59    |253.71    |2026-10-29 12:25:12.225509|
|P00003    |Cedar Wellness 0003  |Harbor|Beauty        |Wellness   |2024-03-07  |true     |27.95     |51.28     |2026-10-29 12:25:12.225509|
|P00004    |Harbor Audio 0004    |Harbor|Electronics   |Audio      |2021-10-07  |true     |152.65    |180.64    |2026-10-29 12:25:12.225509|
|P00005    |C

### Validate Gold Product Dimension

In [7]:
gold_product_count = df_gold_product.count()

expected_columns = GOLD_PRODUCT_COLUMNS + [
    "_gold_processed_timestamp"
]

assert df_gold_product.columns == expected_columns, (
    f"Unexpected schema. Expected {expected_columns}, "
    f"received {df_gold_product.columns}"
)

assert gold_product_count == silver_product_count, (
    f"Row count changed: Silver={silver_product_count}, "
    f"Gold={gold_product_count}"
)

null_product_ids = df_gold_product.filter(
    F.col("product_id").isNull()
).count()

assert null_product_ids == 0, (
    f"Found {null_product_ids} null product IDs"
)

null_gold_timestamps = df_gold_product.filter(
    F.col("_gold_processed_timestamp").isNull()
).count()

assert null_gold_timestamps == 0, (
    f"Found {null_gold_timestamps} null Gold timestamps"
)

print("Gold product validation passed.")
print("Gold product count:", gold_product_count)

Gold product validation passed.
Gold product count: 996


### Write Gold Product Dimension

In [8]:
(
    df_gold_product.write
    .format("parquet")
    .mode("overwrite")
    .save(GOLD_PRODUCT_PATH)
)

print("Gold product dimension written to:")
print(GOLD_PRODUCT_PATH)

Gold product dimension written to:
/content/drive/MyDrive/pyspark-ecommerce-analytics-data/gold/dim_product


### Read Back and Verify

In [9]:
df_gold_product_saved = spark.read.parquet(
    GOLD_PRODUCT_PATH
)

saved_product_count = df_gold_product_saved.count()

assert saved_product_count == gold_product_count, (
    f"Saved count mismatch: expected {gold_product_count}, "
    f"received {saved_product_count}"
)

assert df_gold_product_saved.columns == expected_columns, (
    "Saved Gold schema does not match the expected schema"
)

saved_null_product_ids = df_gold_product_saved.filter(
    F.col("product_id").isNull()
).count()

assert saved_null_product_ids == 0, (
    "Saved Gold dimension contains null product IDs"
)

df_gold_product_saved.printSchema()
df_gold_product_saved.show(10, truncate=False)

print("Read-back verification passed.")

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- is_active: string (nullable = true)
 |-- cost_price: decimal(10,2) (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- _gold_processed_timestamp: timestamp (nullable = true)

+----------+---------------------+------+--------------+-----------+------------+---------+----------+----------+--------------------------+
|product_id|product_name         |brand |category      |subcategory|created_date|is_active|cost_price|unit_price|_gold_processed_timestamp |
+----------+---------------------+------+--------------+-----------+------------+---------+----------+----------+--------------------------+
|P00001    |Orion Wellness 0001  |Nova  |Beauty        |Wellness   |2025-05-01  |true     |14.01     |22.83     |2026-10-29 

### Quality Metrics and Run Summary

In [10]:
summary = {
    "silver_product_count": silver_product_count,
    "gold_product_count": gold_product_count,
    "saved_product_count": saved_product_count,
    "null_product_ids": saved_null_product_ids,
    "columns_in_gold": len(expected_columns),
    "output_path": GOLD_PRODUCT_PATH,
    "validation_status": "PASSED",
}

for metric, value in summary.items():
    print(f"{metric}: {value}")

silver_product_count: 996
gold_product_count: 996
saved_product_count: 996
null_product_ids: 0
columns_in_gold: 10
output_path: /content/drive/MyDrive/pyspark-ecommerce-analytics-data/gold/dim_product
validation_status: PASSED
